# Sanity and information checks: monthly firm variables

Checks every imported time-series firm variable against the filtered U.S. price universe and against each
other. All functions live in `src/datastream/firm_evaluation.py`; this notebook only configures and displays.
Tables are written to `OUTPUT_DIR` as CSV so results can be compared across downloads.

| Block | Question | Main outputs |
|---|---|---|
| **A Coverage** | For which firms and years is the variable there? | EW/VW coverage of the price universe, coverage by size group, firm-years, history gaps, matching |
| **B Timing** | When and how often does the value change, and does it stop when the stock dies? | changes per firm-year, reporting lag after fiscal year end, stale runs, values after the last price |
| **C Values** | Are the values plausible? | yearly quantiles and means, zeros/negatives, robust-z outliers, unit errors (x1000 in several items at once) |
| **D Consistency** | Do variables agree with each other and with market data? | identities and bounds from `config/firm_relations.csv`, recomputed ratios, alignment of update months |

**Prerequisites:** variables imported with `scripts/03_import_firm_variables.py`, the monthly universe built
with `scripts/05_build_monthly_universe.py`, and (for the reporting lag) the static WC05350 imported.

**Which observations are used.** Coverage, history and padding (A, B3) use all firm-months of a variable.
Timing (B1, B2), levels, distributions and jumps (A4, C) and relations (D1) use only firm-months in which the
stock is in the filtered price universe, so padded values after delisting, pre-listing history and securities
outside the universe (non-common stocks, SPAC units, ...) do not distort them.

The summary flags use the heuristic thresholds in `firm_evaluation.THRESHOLDS`; they mark what to look at,
not what is wrong. Values are checked as downloaded (monthly Datastream series, dates moved to month end).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from datastream import firm_evaluation as fe
from datastream.preprocessing.firm_data import VARIABLES_SUBDIR, load_variable, read_registry
from datastream.preprocessing.static_data import load_static, static_panel_path

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
fe.set_style()

# ── configuration ───────────────────────────────────────────────────────────────────────────────────
REGION = "US"
FIRM_ROOT = Path(os.environ.get("DS_FIRM_ROOT", "D:/Datastream/Firmcharacteristics_Monthly")) / REGION
PRICE_PATH = Path(os.environ.get("DS_PRICE_PATH", "D:/Datastream/PriceData/US/processed"))
PENNY_PERCENTILE = "0.2"
UNIVERSE_FILE = PRICE_PATH / f"monthly_universe_{PENNY_PERCENTILE}.parquet"
VARIABLES = None                   # None = all imported variable panels, or e.g. ["WC02999", "WC01001"]
FYE_VARIABLE = "WC05350"           # static fiscal-year end (for the reporting lag)
ALIGNMENT_CATEGORIES = ["balance_sheet", "income_statement", "cash_flow"]   # for D2
COVERAGE_YEAR = None               # reference year for coverage in the summary; None = year before the last
                                   # full year (recent values, ESG in particular, are published with a delay)
OUTPUT_DIR = FIRM_ROOT / "Paneldata" / "checks"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
REGISTRY_FILE = REPO / "config" / "firm_variables.csv"
RELATIONS_FILE = REPO / "config" / "firm_relations.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and run the checks

In [ ]:
uni = pd.read_parquet(UNIVERSE_FILE)
print(f"Universe: {len(uni):,} stock-months, {uni['DSCD'].nunique():,} stocks, "
      f"{uni['Date'].min():%Y-%m} to {uni['Date'].max():%Y-%m}")

registry = read_registry(REGISTRY_FILE)
signs = dict(zip(registry["variable"], registry.get("sign", pd.Series("", index=registry.index))))
categories = dict(zip(registry["variable"], registry.get("category", pd.Series("", index=registry.index))))
names = dict(zip(registry["variable"], registry.get("name", pd.Series("", index=registry.index))))

available = sorted(p.stem for p in (FIRM_ROOT / VARIABLES_SUBDIR).glob("*.parquet"))
variables = [v for v in (VARIABLES or available) if v in available]
print(f"{len(variables)} variables: {variables}")

fye = None
if static_panel_path(FIRM_ROOT, FYE_VARIABLE).exists():
    fye = fe.fye_months(load_static(FIRM_ROOT, FYE_VARIABLE), FYE_VARIABLE)
    print(f"Fiscal-year end available for {len(fye):,} firms")
else:
    print(f"{FYE_VARIABLE} not imported: reporting lag is skipped")

results, skipped = {}, []
for v in variables:
    r = fe.check_variable(load_variable(FIRM_ROOT, v), v, uni, fye, signs.get(v, ""), COVERAGE_YEAR)
    if r is None:
        skipped.append(v)
        continue
    results[v] = r
    print(f"  {v}: done")
if skipped:
    print(f"Skipped (no observations; e.g. leftovers of earlier imports or date-valued series): {skipped}")

unit_candidates = fe.unit_error_candidates({v: r["jumps"] for v, r in results.items()})

## 2. Summary

One row per variable. `flags` lists the checks outside the thresholds below; the sections after this one
show the details behind each number.

In [ ]:
summary = fe.add_unit_errors(fe.summary_table([r["summary"] for r in results.values()]), unit_candidates)
summary.insert(0, "category", [categories.get(v, "") for v in summary.index])
summary.insert(0, "name", [names.get(v, "") for v in summary.index])
print("Thresholds:", fe.THRESHOLDS)
print(f"Coverage reference year: {summary['coverage_ref_year'].iloc[0]}")
display(summary[["name", "category", "n_flags", "flags", "first_date", "n_firms_in_universe", "first_year_vw_cov_50",
                 "ew_coverage_ref", "vw_coverage_ref", "vw_coverage_last_year", "median_changes_per_year",
                 "modal_lag_months", "share_dead_firms_padded", "share_stale_while_trading",
                 "share_sign_violations", "share_extreme", "n_unit_errors"]])

## A. Coverage
### A1. Coverage of the price universe over time
EW: share of the stocks in the filtered price universe that have a value in that month. VW: their share of
total market cap. A large gap between the two lines means small stocks are missing.

In [ ]:
fig = fe.plot_coverage(results)
fig.savefig(OUTPUT_DIR / "A1_coverage.png", dpi=130, bbox_inches="tight")
plt.show()

### A2. Coverage by size group
Average monthly EW coverage per year and size group (Q1 = smallest; NYSE breakpoints when the statics contain
exchange codes). Shows in which years a variable is effectively a large-cap variable.

In [ ]:
fig = fe.plot_coverage_by_size(results)
fig.savefig(OUTPUT_DIR / "A2_coverage_by_size.png", dpi=130, bbox_inches="tight")
plt.show()

### A3. Firm-years per year and history, matching with the price universe
First table: firms in the price universe with a value, per year. Second table: `n_firms` counts all DSCDs with
a value (also outside the universe), `n_firms_in_universe` those that are ever in the universe with a value.

In [ ]:
firms_per_year = pd.DataFrame({v: r["yearly"]["n_firms"] for v, r in results.items() if len(r["yearly"])}).astype("Int64")
display(firms_per_year.T)

cols = ["n_firms", "n_firms_in_universe", "median_history_months", "share_firms_with_gaps", "share_gap_months",
        "share_universe_firms_ever_covered", "share_variable_firms_in_universe", "outside_universe_share"]
display(summary[cols])

### A4. Levels over time
Median and 1%-trimmed mean of the firm-year values (last value of each firm in each calendar year). A step
in the median that is not visible in the economy points to a change in units, definitions or coverage.

In [ ]:
fig = fe.plot_yearly_levels(results)
plt.show()

## B. Timing
### B1. Value changes per firm-year
Firm-years with at least 12 months in the price universe. Annual items should change about once a year, interim
items up to four times. Firm-years with 0 changes are stale data, or a value that is legitimately constant
(e.g. zero debt, no controversies): compare with B3.

In [ ]:
fig = fe.plot_update_frequency(results)
fig.savefig(OUTPUT_DIR / "B1_update_frequency.png", dpi=130, bbox_inches="tight")
plt.show()

### B2. Reporting lag after fiscal year end
Month of each value change relative to the fiscal-year-end month (current WC05350). The mode is the typical
delay between fiscal year end and availability in Datastream, which matters for point-in-time merges.

In [ ]:
lag = pd.DataFrame({v: r["lag"]["share"] for v, r in results.items() if len(r["lag"])}).T
if len(lag):
    display(lag.reindex(columns=range(12)).fillna(0).style.format("{:.2f}").background_gradient(axis=1, cmap="Blues"))

### B3. Stale values and values after the last price
* `share_after_last_price`: observations after the stock's last month in the filtered price universe.
* `share_dead_firms_padded`: delisted stocks with values more than 12 months after their last price.
* `share_before_first_price`: observations before the stock's first month in the universe (pre-listing history
  added by the data vendor, or months removed by the price filters).
* `share_stale_while_trading`: share of the universe firm-months whose value is **non-zero** and unchanged for
  more than `STALE_MONTHS` months. `share_zero_runs_while_trading`: the same for runs of zeros, which are usually
  economic (no debt, no dividends, no controversies) and not counted as stale.

Padding and pre-listing values disappear when firm data are merged onto the price universe; the stale share is
what would remain in a merged panel.

In [ ]:
display(summary[["share_after_last_price", "share_after_delisting", "share_dead_firms_padded",
                 "share_before_first_price", "share_stale_while_trading", "share_zero_runs_while_trading",
                 "share_firms_with_stale_run", "share_firm_years_no_change"]])

## C. Values
### C1. Cross-sectional distribution per year
Bands: p5-p95 and p25-p75 of firm-year values, line: median. Log scale for non-negative variables.

In [ ]:
fig = fe.plot_quantiles(results, signs)
fig.savefig(OUTPUT_DIR / "C1_quantiles.png", dpi=130, bbox_inches="tight")
plt.show()

### C2. Zeros, negatives and extreme values
Sign violations are counted for variables marked `nonneg` in the registry. Extreme: |robust z| > `EXTREME_Z`
within the year (median/MAD, on log10 for non-negative variables and on asinh(x / median |x|) otherwise).

In [ ]:
display(summary[["share_zero", "share_negative", "share_sign_violations", "share_extreme"]])
extremes = pd.concat({v: r["extremes"] for v, r in results.items() if len(r["extremes"])},
                     names=["variable"]).reset_index(level=0)
extremes.to_csv(OUTPUT_DIR / "C2_extreme_values.csv", index=False)
display(extremes.sort_values("robust_z", key=abs, ascending=False).head(20))

### C3. Large jumps and unit errors
For non-negative level variables (registry `sign = nonneg`), within the universe: a change by a factor of at
least ~316 (`jump`), or by at least ~8 that is undone within two years (`reversal`). On their own these are
mostly economic: SPACs (pre-IPO balance sheet to trust account), mergers, and cash or short-term debt moving by
orders of magnitude. A **unit error** scales all items of a firm-year by the same factor, so it is identified
across items: at least `UNIT_ERROR_MIN_VARS` items jump in the same month by (nearly) the same power of 1000.

In [ ]:
jump_tables = {v: r["jumps"] for v, r in results.items() if len(r["jumps"])}
if jump_tables:
    jumps = pd.concat(jump_tables, names=["variable"]).reset_index(level=0).reset_index(drop=True)
    jumps.to_csv(OUTPUT_DIR / "C3_large_jumps.csv", index=False)
    print("Large jumps per variable and type:")
    display(jumps.groupby(["variable", "type"]).size().unstack(fill_value=0))
    print("Firm-months with jumps in several items, by year (SPAC waves show up here):")
    multi = unit_candidates.assign(Year=unit_candidates["Date"].dt.year)
    display(multi.groupby(["Year", "unit_error"]).size().unstack(fill_value=0).T)
unit_candidates.to_csv(OUTPUT_DIR / "C3_multi_item_jumps.csv", index=False)
unit_errors = unit_candidates[unit_candidates["unit_error"]]
print(f"Unit error candidates: {len(unit_errors)} firm-months")
display(unit_errors.head(30))

## D. Consistency across variables
### D1. Identities, bounds and recomputed ratios
Evaluated on December cross-sections of firm-years in the price universe. `share_ratio_within_5pct` is computed
over firm-years with a non-zero right-hand side; 0 = 0 cases are shown in `share_both_zero`. `compare` rows are
informational: the
definitions differ (e.g. average vs. year-end equity), so look at the median ratio and rank correlation.
The `mtbv` row also checks units: MV is in millions, Worldscope items are assumed to be in thousands.
Relations with variables that are not imported yet are skipped.

In [ ]:
relations = fe.read_relations(RELATIONS_FILE)
needed = sorted(set().union(*[fe._names(e) for e in relations[["lhs", "rhs", "condition"]].to_numpy().ravel()]))
dec = fe.december_panel(FIRM_ROOT, [v for v in needed if v in available], uni)
rel_table, rel_by_year, rel_examples = fe.evaluate_relations(dec, relations)
rel_table.to_csv(OUTPUT_DIR / "D1_relations.csv", index=False)
display(rel_table[["relation", "kind", "status", "n_firm_years", "share_violations", "median_ratio",
                   "share_ratio_within_5pct", "share_both_zero", "spearman", "condition", "note"]])

In [ ]:
if len(rel_by_year.columns):
    fig = fe.plot_relation_violations(rel_by_year)
    fig.savefig(OUTPUT_DIR / "D1_violations_by_year.png", dpi=130, bbox_inches="tight")
    plt.show()
for name, ex in rel_examples.items():
    ex.to_csv(OUTPUT_DIR / f"D1_examples_{name}.csv", index=False)

### D2. Alignment of update months
For each pair of accounting variables: share of firm-years (both with exactly one change) in which the value
changes in the same month. Items from the same annual report should be close to 1; low values mean the
series are stamped differently, which matters when variables are combined into ratios.

In [ ]:
align_vars = [v for v in variables if categories.get(v, "") in ALIGNMENT_CATEGORIES]
if len(align_vars) >= 2:
    alignment = fe.update_alignment({v: results[v]["panel_universe"] for v in align_vars})
    alignment.to_csv(OUTPUT_DIR / "D2_update_alignment.csv")
    fig = fe.plot_matrix(alignment, "Share of firm-years with the update in the same month")
    fig.savefig(OUTPUT_DIR / "D2_update_alignment.png", dpi=130, bbox_inches="tight")
    plt.show()

## 3. Save tables

In [ ]:
summary.to_csv(OUTPUT_DIR / "summary.csv")
firms_per_year.to_csv(OUTPUT_DIR / "A3_firms_per_year.csv")
pd.concat({v: r["coverage"] for v, r in results.items()}, names=["variable"]).to_csv(OUTPUT_DIR / "A1_coverage.csv")
pd.concat({v: r["coverage_by_size"] for v, r in results.items() if len(r["coverage_by_size"])},
          names=["variable"]).to_csv(OUTPUT_DIR / "A2_coverage_by_size.csv")
for name, key in [("A4_yearly", "yearly"), ("B1_updates", "updates"), ("C1_quantiles", "quantiles")]:
    tabs = {v: r[key] for v, r in results.items() if len(r[key])}
    if tabs:
        pd.concat(tabs, names=["variable"]).to_csv(OUTPUT_DIR / f"{name}.csv")
if len(lag):
    lag.to_csv(OUTPUT_DIR / "B2_reporting_lag.csv")
print(f"Tables and figures written to {OUTPUT_DIR}")